# Comprehensive EDA of Brazilian Human Rights Hotline Complaints

## Step 6: Temporal Analysis

This notebook examines how recorded human rights hotline complaints change over time. It covers temporal coverage, reporting volume, seasonality, day-of-week and hour patterns, rolling trends, changes in complaint composition, temporal data quality, unusual periods, and candidate structural breaks.

### Main questions

1. What period does the dataset cover, and are there gaps?
2. How does reporting volume evolve by day, week, month, quarter, and year?
3. Are there recurring monthly, weekly, or hourly patterns?
4. Do emergency reports, recorded violations, vulnerable groups, and intake channels change over time?
5. Does completeness change over time, suggesting changes in forms or collection procedures?
6. Which periods have unusually high or low reporting volume?
7. Are apparent changes robust to partial months and choice of counting unit?

> Temporal changes in hotline records may reflect changes in occurrence, awareness, campaigns, access, channel availability, staffing, classification, system migration, or reporting behavior. They should not automatically be interpreted as changes in population incidence.


## 6.1 Imports and analytical configuration

The notebook reads the cleaned dataset produced by Step 3. It defaults to record-level analysis. Use complaint mode only after confirming that `hash` is the appropriate unit.


In [1]:
from pathlib import Path
import calendar
import json
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import linregress
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 160)

PARQUET_PATH = Path("outputs/step_03_data_cleaning/disque100_clean.parquet")
CSV_PATH = Path("outputs/step_03_data_cleaning/disque100_clean.csv")
OUTPUT_DIR = Path("outputs/step_06_temporal_analysis")
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

DATE_COLUMN = "data_de_cadastro"
IDENTIFIER = "hash"
COUNTING_MODE = "record"  # record or complaint
PRIMARY_FREQUENCY = "MS"  # month start
ROLLING_WINDOW_MONTHS = 3
ANOMALY_Z_THRESHOLD = 2.5
MIN_PERIOD_COUNT = 20
TOP_N_CATEGORIES = 8
ALPHA = 0.05
EXPORT_FIGURES_HTML = True
PLOTLY_TEMPLATE = "plotly_white"

# Optional documented events. Add dates only when supported by the research protocol.
EVENTS = [
    # {"date": "2024-05-01", "label": "Campaign or policy event"},
]


## 6.2 Load the cleaned data and establish the counting unit


In [2]:
def load_clean_dataset():
    if PARQUET_PATH.exists():
        return pd.read_parquet(PARQUET_PATH), PARQUET_PATH
    if CSV_PATH.exists():
        return pd.read_csv(CSV_PATH, low_memory=False), CSV_PATH
    raise FileNotFoundError("Run Step 3 or update PARQUET_PATH/CSV_PATH.")


df, DATA_SOURCE = load_clean_dataset()
if DATE_COLUMN not in df.columns:
    raise KeyError(f"Required date column not found: {DATE_COLUMN}")

# Reparse to protect against CSV type loss.
df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN], errors="coerce")
if COUNTING_MODE not in {"record", "complaint"}:
    raise ValueError("COUNTING_MODE must be 'record' or 'complaint'.")
if COUNTING_MODE == "complaint":
    if IDENTIFIER not in df.columns:
        raise ValueError("Complaint mode requires hash.")
    df_analysis = (
        df.dropna(subset=[IDENTIFIER])
          .sort_values(DATE_COLUMN)
          .drop_duplicates(IDENTIFIER, keep="first")
          .copy()
    )
    unit_label = "complaints"
else:
    df_analysis = df.copy()
    unit_label = "records"

print("Source:", DATA_SOURCE.resolve())
print(f"Analytical population: {len(df_analysis):,} {unit_label}")


Source: C:\Projetos\eda-disque100-completo\eda\outputs\step_03_data_cleaning\disque100_clean.parquet
Analytical population: 8,401 records


## 6.3 Temporal validity, coverage, and precision

This section checks missing and future dates, coverage length, repeated timestamps, time-of-day availability, and the number of observations at the temporal boundaries.


In [3]:
valid_dates = df_analysis[DATE_COLUMN].dropna()
future_mask = df_analysis[DATE_COLUMN].notna() & (df_analysis[DATE_COLUMN] > pd.Timestamp.now())
midnight_share = (
    ((valid_dates.dt.hour == 0) & (valid_dates.dt.minute == 0) & (valid_dates.dt.second == 0)).mean()
    if len(valid_dates) else np.nan
)
coverage = pd.DataFrame({
    "metric": [
        "Analytical observations", "Valid dates", "Missing dates", "Future dates",
        "Minimum date", "Maximum date", "Calendar days covered",
        "Unique timestamps", "Exact midnight share"
    ],
    "value": [
        len(df_analysis), len(valid_dates), df_analysis[DATE_COLUMN].isna().sum(),
        future_mask.sum(), valid_dates.min(), valid_dates.max(),
        (valid_dates.max().normalize()-valid_dates.min().normalize()).days+1 if len(valid_dates) else 0,
        valid_dates.nunique(), round(midnight_share, 4) if pd.notna(midnight_share) else np.nan
    ]
})
display(coverage)


,metric,value
0,Analytical observations,8401
1,Valid dates,8401
2,Missing dates,0
3,Future dates,0
4,Minimum date,2021-01-05 08:28:00
5,Maximum date,2025-12-30 15:31:58
6,Calendar days covered,1821
7,Unique timestamps,640
8,Exact midnight share,0.0


## 6.4 Create temporal features

Brazilian Portuguese labels are used for months and weekdays. ISO week and quarter fields support aggregation without altering the original date.


In [4]:
MONTH_NAMES_PT = {
    1:"JAN", 2:"FEV", 3:"MAR", 4:"ABR", 5:"MAI", 6:"JUN",
    7:"JUL", 8:"AGO", 9:"SET", 10:"OUT", 11:"NOV", 12:"DEZ"
}
WEEKDAY_NAMES_PT = {
    0:"SEG", 1:"TER", 2:"QUA", 3:"QUI", 4:"SEX", 5:"SÁB", 6:"DOM"
}

valid = df_analysis.dropna(subset=[DATE_COLUMN]).copy()
valid["date"] = valid[DATE_COLUMN].dt.floor("D")
valid["year"] = valid[DATE_COLUMN].dt.year.astype("Int64")
valid["quarter"] = valid[DATE_COLUMN].dt.to_period("Q").astype("string")
valid["month_start"] = valid[DATE_COLUMN].dt.to_period("M").dt.to_timestamp()
valid["year_month"] = valid[DATE_COLUMN].dt.to_period("M").astype("string")
valid["month_number"] = valid[DATE_COLUMN].dt.month.astype("Int64")
valid["month_name"] = valid["month_number"].map(MONTH_NAMES_PT)
valid["iso_year"] = valid[DATE_COLUMN].dt.isocalendar().year.astype("Int64")
valid["iso_week"] = valid[DATE_COLUMN].dt.isocalendar().week.astype("Int64")
valid["week_start"] = valid[DATE_COLUMN].dt.to_period("W-MON").start_time
valid["weekday_number"] = valid[DATE_COLUMN].dt.dayofweek.astype("Int64")
valid["weekday_name"] = valid["weekday_number"].map(WEEKDAY_NAMES_PT)
valid["hour"] = valid[DATE_COLUMN].dt.hour.astype("Int64")
valid["is_weekend"] = valid["weekday_number"].isin([5,6])
valid["day_of_year"] = valid[DATE_COLUMN].dt.dayofyear.astype("Int64")
valid["days_in_month"] = valid[DATE_COLUMN].dt.days_in_month.astype("Int64")
display(valid[[DATE_COLUMN,"year","quarter","year_month","month_name","week_start","weekday_name","hour"]].head())


AttributeError: 'Series' object has no attribute 'start_time'

## 6.5 Identify complete and partial boundary months

Partial first or last months can create misleading drops. The notebook flags a boundary month as partial when the observed coverage misses the beginning or end of that calendar month.


In [ ]:
if len(valid):
    first_date = valid[DATE_COLUMN].min().normalize()
    last_date = valid[DATE_COLUMN].max().normalize()
    first_month_start = first_date.to_period("M").start_time
    first_month_end = first_date.to_period("M").end_time.normalize()
    last_month_start = last_date.to_period("M").start_time
    last_month_end = last_date.to_period("M").end_time.normalize()
    boundary_status = pd.DataFrame({
        "boundary": ["first_month", "last_month"],
        "month": [str(first_date.to_period("M")), str(last_date.to_period("M"))],
        "observed_start": [first_date, last_month_start],
        "observed_end": [first_month_end, last_date],
        "calendar_start": [first_month_start, last_month_start],
        "calendar_end": [first_month_end, last_month_end],
        "is_partial": [first_date > first_month_start, last_date < last_month_end]
    })
else:
    boundary_status = pd.DataFrame()
display(boundary_status)


## 6.6 Daily, weekly, monthly, quarterly, and annual volume

The main monthly series includes zero-count months between the minimum and maximum dates, making temporal gaps explicit.


In [ ]:
def complete_count_series(data, date_col, frequency):
    observed = data.set_index(date_col).resample(frequency).size().rename("count")
    if observed.empty:
        return observed.reset_index()
    full_index = pd.date_range(observed.index.min(), observed.index.max(), freq=frequency)
    return observed.reindex(full_index, fill_value=0).rename_axis("period").reset_index()


daily_counts = complete_count_series(valid, DATE_COLUMN, "D")
weekly_counts = complete_count_series(valid, DATE_COLUMN, "W-MON")
monthly_counts = complete_count_series(valid, DATE_COLUMN, "MS")
quarterly_counts = complete_count_series(valid, DATE_COLUMN, "QS")
annual_counts = complete_count_series(valid, DATE_COLUMN, "YS")

monthly_counts["rolling_mean"] = monthly_counts["count"].rolling(ROLLING_WINDOW_MONTHS, min_periods=1, center=True).mean()
monthly_counts["month_over_month_pct"] = monthly_counts["count"].pct_change().replace([np.inf,-np.inf],np.nan)*100
monthly_counts["year_over_year_pct"] = monthly_counts["count"].pct_change(12).replace([np.inf,-np.inf],np.nan)*100

display(monthly_counts)


In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=monthly_counts["period"], y=monthly_counts["count"], mode="lines+markers", name=unit_label.title()))
fig.add_trace(go.Scatter(x=monthly_counts["period"], y=monthly_counts["rolling_mean"], mode="lines", name=f"{ROLLING_WINDOW_MONTHS}-month centered mean", line=dict(width=3)))
for event in EVENTS:
    fig.add_vline(x=pd.Timestamp(event["date"]).timestamp()*1000, line_dash="dash", line_color="gray")
    fig.add_annotation(x=event["date"], y=1, yref="paper", text=event["label"], showarrow=False, textangle=-90)
fig.update_layout(title="Monthly recorded hotline volume", xaxis_title="Month", yaxis_title=unit_label.title(), template=PLOTLY_TEMPLATE)
fig.show()
if EXPORT_FIGURES_HTML: fig.write_html(FIGURE_DIR/"monthly_volume_and_rolling_mean.html", include_plotlyjs=True)


## 6.7 Calendar completeness and zero-count periods

A zero month may indicate no records, system interruption, extraction gaps, or an out-of-scope period. It should be investigated against operational documentation.


In [ ]:
zero_months = monthly_counts[monthly_counts["count"] == 0].copy()
observed_days_by_month = valid.groupby("month_start")["date"].nunique().rename("days_with_records").reset_index()
calendar_completeness = monthly_counts[["period","count"]].merge(
    observed_days_by_month, left_on="period", right_on="month_start", how="left"
).drop(columns="month_start")
calendar_completeness["days_with_records"] = calendar_completeness["days_with_records"].fillna(0).astype(int)
calendar_completeness["calendar_days"] = calendar_completeness["period"].dt.days_in_month
calendar_completeness["days_with_records_pct"] = (calendar_completeness["days_with_records"]/calendar_completeness["calendar_days"]*100).round(2)
display(zero_months)
display(calendar_completeness)

fig = px.bar(calendar_completeness, x="period", y="days_with_records_pct", color="days_with_records_pct", color_continuous_scale="Viridis", title="Share of calendar days containing at least one record", labels={"days_with_records_pct":"Days with records (%)"})
fig.update_layout(template=PLOTLY_TEMPLATE, coloraxis_showscale=False)
fig.show()
if EXPORT_FIGURES_HTML: fig.write_html(FIGURE_DIR/"calendar_day_coverage.html", include_plotlyjs=True)


## 6.8 Month-over-month and year-over-year change

Percentage changes become unstable when the previous period has a small or zero count. The table retains the original count so changes can be interpreted in context.


In [ ]:
change_table = monthly_counts[["period","count","month_over_month_pct","year_over_year_pct"]].copy()
display(change_table)
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=("Month-over-month change", "Year-over-year change"))
fig.add_trace(go.Bar(x=change_table["period"], y=change_table["month_over_month_pct"], name="MoM %"), row=1, col=1)
fig.add_trace(go.Bar(x=change_table["period"], y=change_table["year_over_year_pct"], name="YoY %"), row=2, col=1)
fig.add_hline(y=0, line_color="gray", row=1, col=1)
fig.add_hline(y=0, line_color="gray", row=2, col=1)
fig.update_layout(height=700, title="Temporal percentage changes", template=PLOTLY_TEMPLATE, showlegend=False)
fig.show()
if EXPORT_FIGURES_HTML: fig.write_html(FIGURE_DIR/"monthly_percentage_changes.html", include_plotlyjs=True)


## 6.9 Weekday and hourly reporting patterns

Hour-of-day analysis is meaningful only when timestamps contain real time information. If most values occur exactly at midnight, the notebook marks hourly analysis as unsuitable.


In [ ]:
weekday_order = ["SEG","TER","QUA","QUI","SEX","SÁB","DOM"]
weekday_counts = valid["weekday_name"].value_counts().reindex(weekday_order, fill_value=0).rename_axis("weekday").reset_index(name="count")
weekday_counts["percent"] = (weekday_counts["count"]/weekday_counts["count"].sum()*100).round(2)
display(weekday_counts)
fig = px.bar(weekday_counts, x="weekday", y="count", text="percent", category_orders={"weekday":weekday_order}, title="Recorded volume by weekday", labels={"count":unit_label.title()})
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(template=PLOTLY_TEMPLATE)
fig.show()
if EXPORT_FIGURES_HTML: fig.write_html(FIGURE_DIR/"weekday_distribution.html", include_plotlyjs=True)

HOURLY_ANALYSIS_VALID = pd.notna(midnight_share) and midnight_share < 0.80
if HOURLY_ANALYSIS_VALID:
    hour_counts = valid["hour"].value_counts().reindex(range(24), fill_value=0).rename_axis("hour").reset_index(name="count")
    fig = px.line(hour_counts, x="hour", y="count", markers=True, title="Recorded volume by hour of day", labels={"count":unit_label.title()})
    fig.update_layout(template=PLOTLY_TEMPLATE, xaxis=dict(dtick=1))
    fig.show()
    if EXPORT_FIGURES_HTML: fig.write_html(FIGURE_DIR/"hourly_distribution.html", include_plotlyjs=True)
else:
    hour_counts = pd.DataFrame()
    display(Markdown("**Hourly analysis skipped:** at least 80% of valid timestamps occur exactly at midnight, suggesting date-only precision or default times."))


## 6.10 Seasonality by calendar month

Monthly seasonality is assessed descriptively by comparing the same calendar month across years. Partial boundary months are noted separately. A formal seasonal model requires enough complete cycles, typically at least three years for stable monthly comparisons.


In [ ]:
seasonal_month = valid.groupby(["year","month_number","month_name"]).size().rename("count").reset_index()
month_order = [MONTH_NAMES_PT[i] for i in range(1,13)]
seasonal_summary = seasonal_month.groupby(["month_number","month_name"])["count"].agg(["count","mean","median","std","min","max"]).reset_index().sort_values("month_number")
display(seasonal_summary)

fig = px.box(seasonal_month, x="month_name", y="count", points="all", category_orders={"month_name":month_order}, title="Distribution of monthly reporting volume across years", labels={"count":unit_label.title(),"month_name":"Calendar month"})
fig.update_layout(template=PLOTLY_TEMPLATE)
fig.show()
if EXPORT_FIGURES_HTML: fig.write_html(FIGURE_DIR/"calendar_month_seasonality.html", include_plotlyjs=True)

seasonal_pivot = seasonal_month.pivot(index="year", columns="month_name", values="count").reindex(columns=month_order)
fig = px.imshow(seasonal_pivot, text_auto=True, aspect="auto", color_continuous_scale="Blues", title="Monthly volume by year", labels={"color":unit_label.title()})
fig.update_layout(template=PLOTLY_TEMPLATE)
fig.show()
if EXPORT_FIGURES_HTML: fig.write_html(FIGURE_DIR/"year_month_heatmap.html", include_plotlyjs=True)


## 6.11 Long-term linear trend

A simple linear model is used only as a descriptive summary of direction. Serial correlation, seasonality, system changes, and nonlinear dynamics mean that ordinary regression p-values should not be treated as definitive time-series inference.


In [ ]:
trend_data = monthly_counts.copy()
trend_data["time_index"] = np.arange(len(trend_data))
if len(trend_data) >= 3:
    trend_fit = linregress(trend_data["time_index"], trend_data["count"])
    trend_data["linear_fit"] = trend_fit.intercept + trend_fit.slope*trend_data["time_index"]
    trend_summary = pd.DataFrame({
        "metric":["Monthly slope","Intercept","R-squared","Naive p-value","Standard error"],
        "value":[trend_fit.slope,trend_fit.intercept,trend_fit.rvalue**2,trend_fit.pvalue,trend_fit.stderr]
    })
    display(trend_summary)
    fig = px.scatter(trend_data, x="period", y="count", title="Monthly volume with descriptive linear trend")
    fig.add_trace(go.Scatter(x=trend_data["period"], y=trend_data["linear_fit"], mode="lines", name="Linear trend"))
    fig.update_layout(template=PLOTLY_TEMPLATE)
    fig.show()
    if EXPORT_FIGURES_HTML: fig.write_html(FIGURE_DIR/"descriptive_linear_trend.html", include_plotlyjs=True)
else:
    trend_summary = pd.DataFrame()


## 6.12 Robust anomaly detection

Monthly anomalies are flagged using a rolling median and median absolute deviation (MAD). This is more robust than a global z-score when the series contains spikes. Flags are candidates for investigation, not proof of unusual incidence.


In [ ]:
anomaly_table = monthly_counts[["period","count"]].copy()
window = min(12, max(3, len(anomaly_table)))
anomaly_table["rolling_median"] = anomaly_table["count"].rolling(window, center=True, min_periods=3).median()
anomaly_table["absolute_deviation"] = (anomaly_table["count"]-anomaly_table["rolling_median"]).abs()
anomaly_table["rolling_mad"] = anomaly_table["absolute_deviation"].rolling(window, center=True, min_periods=3).median()
anomaly_table["robust_z"] = 0.6745*(anomaly_table["count"]-anomaly_table["rolling_median"])/anomaly_table["rolling_mad"].replace(0,np.nan)
anomaly_table["is_anomaly"] = anomaly_table["robust_z"].abs() >= ANOMALY_Z_THRESHOLD
anomaly_table["direction"] = np.where(anomaly_table["robust_z"]>0,"high",np.where(anomaly_table["robust_z"]<0,"low","none"))
display(anomaly_table[anomaly_table["is_anomaly"]])

fig = go.Figure()
fig.add_trace(go.Scatter(x=anomaly_table["period"], y=anomaly_table["count"], mode="lines+markers", name="Monthly count"))
flags = anomaly_table[anomaly_table["is_anomaly"]]
fig.add_trace(go.Scatter(x=flags["period"], y=flags["count"], mode="markers", name="Review flag", marker=dict(size=13,color="red",symbol="diamond"), text=flags["robust_z"], hovertemplate="%{x}<br>count=%{y}<br>robust z=%{text:.2f}<extra></extra>"))
fig.update_layout(title="Robust monthly anomaly flags", template=PLOTLY_TEMPLATE, xaxis_title="Month", yaxis_title=unit_label.title())
fig.show()
if EXPORT_FIGURES_HTML: fig.write_html(FIGURE_DIR/"robust_monthly_anomalies.html", include_plotlyjs=True)


## 6.13 Candidate structural breaks

The notebook identifies large changes in adjacent rolling means as **candidate** breaks. These must be matched to documented system, form, policy, channel, campaign, or extraction changes. This is exploratory and not a formal change-point model.


In [ ]:
break_table = monthly_counts[["period","count"]].copy()
BREAK_WINDOW = 3
break_table["mean_before"] = break_table["count"].rolling(BREAK_WINDOW, min_periods=BREAK_WINDOW).mean().shift(1)
break_table["mean_after"] = break_table["count"][::-1].rolling(BREAK_WINDOW, min_periods=BREAK_WINDOW).mean()[::-1].shift(-1)
break_table["absolute_shift"] = (break_table["mean_after"]-break_table["mean_before"]).abs()
shift_median = break_table["absolute_shift"].median()
shift_mad = (break_table["absolute_shift"]-shift_median).abs().median()
break_table["shift_robust_z"] = 0.6745*(break_table["absolute_shift"]-shift_median)/(shift_mad if shift_mad else np.nan)
break_table["candidate_break"] = break_table["shift_robust_z"] >= ANOMALY_Z_THRESHOLD
candidate_breaks = break_table[break_table["candidate_break"]].sort_values("absolute_shift",ascending=False)
display(candidate_breaks)


## 6.14 Temporal composition of priority categorical variables

Counts can increase while category proportions remain stable, or total volume can remain stable while composition changes. This section displays monthly shares for the most frequent categories.


In [ ]:
TEMPORAL_CATEGORICALS = [
    "violacao", "grupo_vulneravel", "canal_de_atendimento",
    "denuncia_emergencial", "genero_da_vitima",
    "faixa_etaria_da_vitima", "relacao_vitima_suspeito", "uf"
]
composition_summaries = []
for column in [c for c in TEMPORAL_CATEGORICALS if c in valid.columns]:
    top_categories = valid[column].dropna().value_counts().head(TOP_N_CATEGORIES).index
    temp = valid[["month_start",column]].dropna().copy()
    temp[column] = temp[column].where(temp[column].isin(top_categories), "OTHER")
    counts = temp.groupby(["month_start",column]).size().rename("count").reset_index()
    counts["monthly_total"] = counts.groupby("month_start")["count"].transform("sum")
    counts["share_pct"] = counts["count"]/counts["monthly_total"]*100
    counts.to_csv(TABLE_DIR/f"temporal_composition_{column}.csv",index=False,encoding="utf-8-sig")
    fig = px.area(counts, x="month_start", y="share_pct", color=column, groupnorm=None, title=f"Monthly composition: {column}", labels={"share_pct":"Share of valid monthly observations (%)"})
    fig.update_layout(template=PLOTLY_TEMPLATE, yaxis_range=[0,100])
    fig.show()
    if EXPORT_FIGURES_HTML: fig.write_html(FIGURE_DIR/f"temporal_composition_{column}.html",include_plotlyjs=True)
    first = counts[counts["month_start"]==counts["month_start"].min()][[column,"share_pct"]].rename(columns={"share_pct":"first_share"})
    last = counts[counts["month_start"]==counts["month_start"].max()][[column,"share_pct"]].rename(columns={"share_pct":"last_share"})
    change = first.merge(last,on=column,how="outer").fillna(0)
    change["share_change_pp"] = change["last_share"]-change["first_share"]
    change["variable"] = column
    composition_summaries.append(change.rename(columns={column:"category"}))
composition_change = pd.concat(composition_summaries,ignore_index=True) if composition_summaries else pd.DataFrame()
display(composition_change.sort_values("share_change_pp",key=abs,ascending=False).head(50) if not composition_change.empty else composition_change)


## 6.15 Temporal data quality

Changes in missingness can indicate redesigned forms, new mandatory fields, system migration, or altered collection practices. Such changes can mimic substantive changes in victim or suspect profiles.


In [ ]:
QUALITY_COLUMNS = [
    "violacao", "motivacao", "genero_da_vitima", "faixa_etaria_da_vitima",
    "raca_cor_da_vitima", "genero_do_suspeito", "faixa_etaria_do_suspeito",
    "raca_cor_do_suspeito", "uf", "municipio"
]
quality_rows = []
for column in [c for c in QUALITY_COLUMNS if c in valid.columns]:
    temp = valid[["month_start",column]].copy()
    summary = temp.groupby("month_start")[column].agg(total="size",missing=lambda s:s.isna().sum()).reset_index()
    summary["missing_pct"] = summary["missing"]/summary["total"]*100
    summary["column"] = column
    quality_rows.append(summary)
temporal_quality = pd.concat(quality_rows,ignore_index=True) if quality_rows else pd.DataFrame()
display(temporal_quality.head(50))
if not temporal_quality.empty:
    fig = px.line(temporal_quality,x="month_start",y="missing_pct",color="column",markers=False,title="Missingness over time for priority fields",labels={"missing_pct":"Missing (%)","month_start":"Month"})
    fig.update_layout(template=PLOTLY_TEMPLATE)
    fig.show()
    if EXPORT_FIGURES_HTML: fig.write_html(FIGURE_DIR/"temporal_missingness.html",include_plotlyjs=True)


## 6.16 Emergency-report rate over time

This section calculates the monthly proportion classified as emergency. The positive label must be confirmed against the official domain.


In [ ]:
emergency_monthly = pd.DataFrame()
if "denuncia_emergencial" in valid.columns:
    emergency_keys = valid["denuncia_emergencial"].astype("string").str.upper().str.strip()
    candidate_positive = {"SIM","S","YES","TRUE","1"}
    valid_emergency = emergency_keys.notna()
    temp = valid.loc[valid_emergency,["month_start"]].copy()
    temp["is_emergency"] = emergency_keys[valid_emergency].isin(candidate_positive).to_numpy()
    emergency_monthly = temp.groupby("month_start")["is_emergency"].agg(total="size",emergency="sum",rate="mean").reset_index()
    emergency_monthly["emergency_pct"] = emergency_monthly["rate"]*100
    display(emergency_monthly)
    fig = px.line(emergency_monthly,x="month_start",y="emergency_pct",markers=True,title="Monthly emergency classification rate",labels={"emergency_pct":"Emergency (%)","month_start":"Month"})
    fig.update_layout(template=PLOTLY_TEMPLATE,yaxis_range=[0,100])
    fig.show()
    if EXPORT_FIGURES_HTML: fig.write_html(FIGURE_DIR/"monthly_emergency_rate.html",include_plotlyjs=True)


## 6.17 Sensitivity to counting mode

When `hash` repeats, record counts and unique-complaint counts can show different trends. The comparison below is generated whenever `hash` is available, regardless of the selected primary counting mode.


In [ ]:
counting_mode_comparison = pd.DataFrame()
if IDENTIFIER in df.columns:
    temp = df.dropna(subset=[DATE_COLUMN]).copy()
    temp["month_start"] = temp[DATE_COLUMN].dt.to_period("M").dt.to_timestamp()
    counting_mode_comparison = temp.groupby("month_start").agg(
        records=(IDENTIFIER,"size"),
        unique_hashes=(IDENTIFIER,"nunique")
    ).reset_index()
    counting_mode_comparison["records_per_hash"] = counting_mode_comparison["records"]/counting_mode_comparison["unique_hashes"].replace(0,np.nan)
    display(counting_mode_comparison)
    long = counting_mode_comparison.melt(id_vars="month_start",value_vars=["records","unique_hashes"],var_name="measure",value_name="count")
    fig = px.line(long,x="month_start",y="count",color="measure",markers=True,title="Sensitivity to temporal counting unit")
    fig.update_layout(template=PLOTLY_TEMPLATE)
    fig.show()
    if EXPORT_FIGURES_HTML: fig.write_html(FIGURE_DIR/"counting_mode_sensitivity.html",include_plotlyjs=True)


## 6.18 Automated temporal findings register

The register highlights incomplete boundaries, zero periods, robust anomaly flags, candidate breaks, large year-over-year changes, and notable missingness shifts. Every item requires contextual validation.


In [ ]:
finding_rows=[]
if not boundary_status.empty:
    for _,r in boundary_status[boundary_status["is_partial"]].iterrows():
        finding_rows.append({"finding_type":"partial_boundary","period":r["month"],"finding":f"{r['boundary']} is partial","interpretation":"Exclude or annotate this month in trend comparisons."})
for _,r in zero_months.iterrows():
    finding_rows.append({"finding_type":"zero_month","period":str(r["period"].date()),"finding":"No records in the completed monthly index","interpretation":"Check extraction coverage, system availability, and scope."})
for _,r in anomaly_table[anomaly_table["is_anomaly"]].iterrows():
    finding_rows.append({"finding_type":"robust_anomaly","period":str(r["period"].date()),"finding":f"{r['direction']} monthly volume; robust z={r['robust_z']:.2f}","interpretation":"Compare with campaigns, system changes, operational events, and category composition."})
for _,r in candidate_breaks.iterrows():
    finding_rows.append({"finding_type":"candidate_break","period":str(r["period"].date()),"finding":f"Adjacent rolling means differ by {r['absolute_shift']:.1f}","interpretation":"Validate against documented collection or policy changes before formal modeling."})
for _,r in change_table[change_table["year_over_year_pct"].abs()>=50].iterrows():
    finding_rows.append({"finding_type":"large_yoy_change","period":str(r["period"].date()),"finding":f"Year-over-year change={r['year_over_year_pct']:.1f}%","interpretation":"Inspect denominator, partial periods, reporting channels, and composition."})
temporal_findings = pd.DataFrame(finding_rows)
display(temporal_findings)


## 6.19 Export tables, figures, and metadata


In [ ]:
exports={
    "coverage.csv":coverage,
    "boundary_status.csv":boundary_status,
    "daily_counts.csv":daily_counts,
    "weekly_counts.csv":weekly_counts,
    "monthly_counts.csv":monthly_counts,
    "quarterly_counts.csv":quarterly_counts,
    "annual_counts.csv":annual_counts,
    "calendar_completeness.csv":calendar_completeness,
    "change_table.csv":change_table,
    "weekday_counts.csv":weekday_counts,
    "hour_counts.csv":hour_counts,
    "seasonal_month.csv":seasonal_month,
    "seasonal_summary.csv":seasonal_summary,
    "trend_summary.csv":trend_summary,
    "anomaly_table.csv":anomaly_table,
    "candidate_breaks.csv":candidate_breaks,
    "composition_change.csv":composition_change,
    "temporal_quality.csv":temporal_quality,
    "emergency_monthly.csv":emergency_monthly,
    "counting_mode_comparison.csv":counting_mode_comparison,
    "temporal_findings.csv":temporal_findings,
}
for filename,table in exports.items():
    table.to_csv(TABLE_DIR/filename,index=False,encoding="utf-8-sig")

figure_index=pd.DataFrame({"figure_file":sorted(p.name for p in FIGURE_DIR.glob("*.html"))})
figure_index.to_csv(OUTPUT_DIR/"figure_index.csv",index=False,encoding="utf-8-sig")
metadata={
    "data_source":str(DATA_SOURCE),"date_column":DATE_COLUMN,
    "counting_mode":COUNTING_MODE,"analytical_unit":unit_label,
    "analytical_rows":int(len(df_analysis)),"valid_date_rows":int(len(valid)),
    "coverage_start":str(valid_dates.min()),"coverage_end":str(valid_dates.max()),
    "rolling_window_months":ROLLING_WINDOW_MONTHS,
    "anomaly_method":"rolling median and MAD",
    "anomaly_robust_z_threshold":ANOMALY_Z_THRESHOLD,
    "hourly_analysis_valid":bool(HOURLY_ANALYSIS_VALID),
    "documented_events":EVENTS
}
(OUTPUT_DIR/"run_metadata.json").write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding="utf-8")
print("Tables:",TABLE_DIR.resolve())
print("Interactive figures:",FIGURE_DIR.resolve())


## 6.20 Interpretation checklist

Before reporting temporal findings:

1. State whether counts represent records or unique complaints.
2. Report the exact coverage period and flag partial boundary months.
3. Investigate zero-count months before treating them as true zeros.
4. Distinguish reporting volume from population incidence.
5. Check whether channel availability, campaigns, staffing, or system migrations changed.
6. Review temporal missingness before interpreting changes in victim or suspect composition.
7. Confirm that hourly timestamps contain genuine time information.
8. Compare counts with proportions because composition can change independently of volume.
9. Treat anomaly and break flags as exploratory candidates requiring external documentation.
10. Use population denominators and exposure periods before calculating temporal rates.

### Recommended reporting language

“Recorded hotline volume changed over the observation period, with selected months flagged as unusual relative to a rolling median. These deviations may reflect changes in occurrence, reporting behavior, campaigns, channel availability, or data collection. Accordingly, the temporal results describe recorded reports and are not interpreted as direct estimates of population incidence.”

## Next step

**Step 7: Geographic Analysis** should examine complaint, victim, and suspect locations; validate municipality-UF combinations against official IBGE codes; calculate population-standardized rates using year-aligned denominators; map completeness separately from volume; and suppress small cells where disclosure risk exists.
